# Training Utilities: SGD, Warmup + Cosine LR, Gradient Clipping, EMA — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Every training script has the same scaffolding around `loss.backward()`, whether it trains a ResNet, a diffusion U-Net or an LLM:

- an **optimizer step**, here SGD with momentum, Nesterov momentum and weight decay;
- a **learning-rate schedule**. Linear warmup followed by cosine decay is the default in most modern recipes: MAE and CLIP use it, and LLaMA decays to 10% of the peak LR after 2000 warmup steps;
- **gradient clipping** by global norm. `clip_grad_norm_(params, 1.0)` appears in nearly every transformer recipe;
- an **exponential moving average (EMA)** of the weights. DDPM and Stable Diffusion sample from EMA weights, and the MoCo key encoder and the DINO/BYOL teachers are EMAs of the online network.

MLE and research-engineer interviews often ask you to write these from memory and then probe the details. Where does weight decay go? What does PyTorch store in the momentum buffer? What does `clip_grad_norm_` return? What happens to BatchNorm buffers in an EMA? Your versions must match PyTorch **exactly**: the grader runs them side by side with `torch.optim.SGD` and `torch.nn.utils.clip_grad_norm_`.

---

### Requirements

1. **`SGD(params, lr, momentum=0.0, weight_decay=0.0, nesterov=False)`** with `step()` and `zero_grad()`. Its updates must equal `torch.optim.SGD` (with `dampening=0`). For each parameter $\theta$ with gradient $\nabla$ at step $t = 1, 2, \dots$:

   $$g_t = \nabla + \lambda\,\theta_{t-1} \qquad \text{(weight decay = L2 penalty, added to the gradient)}$$
   $$b_1 = g_1, \qquad b_t = \mu\, b_{t-1} + g_t \quad (t > 1) \qquad \text{(momentum buffer, only if } \mu > 0)$$
   $$d_t = \begin{cases} g_t + \mu\, b_t & \text{nesterov} \\ b_t & \text{otherwise} \end{cases} \qquad (d_t = g_t \text{ when } \mu = 0), \qquad \theta_t = \theta_{t-1} - \eta\, d_t$$

   - `params` may be a **generator** such as `model.parameters()`.
   - Skip parameters whose `.grad` is `None` completely: no update, no weight decay, no buffer.
   - Keep one momentum buffer per parameter. Do not modify `p.grad` inside `step()`, and do not let the buffer share memory with `p.grad`.
   - The learning rate can change between steps: a schedule writes `group["lr"]` in `param_groups` (or `self.lr` for a plain class). Use the current value at every step. $\eta$ multiplies the step direction and never enters the buffer, so the textbook velocity $v \leftarrow \mu v - \eta g$ matches PyTorch only while the LR is constant. The grader changes the LR between steps.
   - The update must not be recorded by autograd.
   - Subclassing `torch.optim.Optimizer` is recommended: it gives you `param_groups`, a per-parameter `self.state` dict and `zero_grad()` for free. A plain class with `step()` and `zero_grad()` is fine too (store the learning rate as `self.lr`).
   - `nesterov=True` with `momentum=0` should raise `ValueError`, as PyTorch does (not graded).

2. **`warmup_cosine_lr(step, total_steps, warmup_steps, base_lr, min_lr=0.0) -> float`**, with a 0-indexed step $s$. Write $W$ = `warmup_steps`, $T$ = `total_steps`, $\eta_{\max}$ = `base_lr`, $\eta_{\min}$ = `min_lr`:

   $$\eta(s) = \begin{cases}
   \eta_{\max}\,\dfrac{s+1}{W} & s < W \\
   \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\bigl(1+\cos(\pi p)\bigr), \quad p = \dfrac{s-W}{T-W} & W \le s < T \\
   \eta_{\min} & s \ge T
   \end{cases}$$

   - **Convention:** step 0 already uses `base_lr / warmup_steps`, not 0, so no step is wasted at LR 0. The last warmup step reaches `base_lr`. (Hugging Face's `get_cosine_schedule_with_warmup` uses `step / warmup` instead and starts at exactly 0.)
   - `warmup_steps = 0` means no warmup, so `lr(0) = base_lr`. Past `total_steps` the LR stays at `min_lr`.

3. **`clip_grad_norm_(params, max_norm) -> Tensor`** must match `torch.nn.utils.clip_grad_norm_`:
   - $\|g\| = \sqrt{\sum_{p}\sum_i g_{p,i}^2}$ is **one global L2 norm** over the gradients of all parameters together. Parameters with `grad=None` are ignored.
   - `clip_coef = max_norm / (total_norm + 1e-6)`. If `clip_coef < 1`, multiply **every** gradient in place by it. Otherwise leave the gradients untouched (clip, don't normalise).
   - Return the total norm **before** clipping (a 0-dim tensor; a float is accepted). If no parameter has a gradient, return 0. All-zero gradients must not turn into NaN.
   - `params` may be a generator.

4. **`EMA(model, decay)`** with `update(model)` and `copy_to(model)`:
   - The constructor takes a snapshot: shadow = a copy of the current parameters and buffers.
   - `update(model)`: for every parameter, $\text{shadow} \leftarrow \text{decay}\cdot\text{shadow} + (1-\text{decay})\cdot\theta$. **Buffers** (e.g. BatchNorm running stats, including the integer `num_batches_tracked`) are **copied**, not averaged.
   - `copy_to(model)` writes the shadow parameters and buffers into `model` **in place**. The model's parameters must remain the same trainable `nn.Parameter` objects.
   - The shadow must not share memory with the model and must not carry an autograd graph.
   - After $k$ updates with weights $\theta_1, \dots, \theta_k$ (and $\theta_0$ at construction): $\text{shadow}_k = \text{decay}^k\,\theta_0 + (1-\text{decay})\sum_{i=1}^{k}\text{decay}^{\,k-i}\,\theta_i$.

---

### Constraints

- ✅ Plain PyTorch tensor ops. You may subclass `torch.optim.Optimizer`.
- ❌ Do not call `torch.optim.SGD` (or `torch.optim._functional`), `torch.nn.utils.clip_grad_norm_`, `torch.optim.lr_scheduler`, or `torch.optim.swa_utils`. The test cells use them only as oracles.
- ✅ Everything runs on CPU in milliseconds.

---

<details>
  <summary>💡 Hint</summary>

  - Write the update rule out before coding. PyTorch's momentum buffer has **no** `(1 - momentum)` factor, and the learning rate multiplies the buffer only at the very end.
  - Weight decay is folded into the gradient *before* the momentum buffer sees it. What does that mean for a parameter whose `.grad` is `None`?
  - `model.parameters()` can be iterated only once. Both your optimizer and your clipping function iterate their params more than once.
  - Optimizer updates and EMA updates are not part of the model's computation graph. Which context manager keeps autograd out of them?
  - For the schedule, sketch lr(step) on paper and mark `warmup_steps` and `total_steps`. At which step does the cosine start, and what is its phase there?

</details>

---

In [ ]:
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# A tiny network used by the tests. `unused` never takes part in forward(), so its
# .grad stays None: optimizers and clipping must skip it.
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 8)
        self.fc2 = nn.Linear(8, 3)
        self.unused = nn.Parameter(torch.randn(4))

    def forward(self, x):
        return self.fc2(torch.tanh(self.fc1(x)))


def make_net(seed=0):
    torch.manual_seed(seed)
    return TinyNet()


print("parameters:", [(n, tuple(p.shape)) for n, p in make_net().named_parameters()])

In [ ]:
class SGD(torch.optim.Optimizer):
    """SGD with momentum / Nesterov / L2 weight decay, matching torch.optim.SGD (dampening=0).

    Per parameter p with gradient g (parameters whose .grad is None are skipped entirely):
        g   <- g + weight_decay * p                           (L2 penalty, added to the gradient)
        buf <- g                   on the first step           (only when momentum > 0)
        buf <- momentum * buf + g  afterwards
        d   <- g + momentum * buf  if nesterov  else  buf      (d = g when momentum == 0)
        p   <- p - lr * d
    """

    def __init__(self, params, lr, momentum=0.0, weight_decay=0.0, nesterov=False):
        if lr < 0 or momentum < 0 or weight_decay < 0:
            raise ValueError("lr, momentum and weight_decay must be non-negative")
        if nesterov and momentum <= 0:
            raise ValueError("Nesterov momentum requires momentum > 0")
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=nesterov)
        # Optimizer.__init__ list()s the params (model.parameters() is a one-shot generator)
        # and provides param_groups, the per-parameter self.state dict and zero_grad().
        super().__init__(params, defaults)

    @torch.no_grad()                      # the update itself must not be recorded by autograd
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu = group["lr"], group["momentum"]
            wd, nesterov = group["weight_decay"], group["nesterov"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)                # out of place: never mutate p.grad
                if mu != 0:
                    state = self.state[p]
                    buf = state.get("momentum_buffer")
                    if buf is None:
                        # clone(): a buffer aliasing p.grad would be wiped by zero_grad()
                        buf = state["momentum_buffer"] = g.clone()
                    else:
                        buf.mul_(mu).add_(g)
                    g = g.add(buf, alpha=mu) if nesterov else buf
                p.add_(g, alpha=-lr)
        return loss

In [ ]:
def sgd_trajectory(opt_cls, steps=20, **kw):
    net = make_net()
    opt = opt_cls(net.parameters(), lr=0.05, **kw)      # a generator, like real code
    g = torch.Generator().manual_seed(1)
    for _ in range(steps):
        x, y = torch.randn(6, 5, generator=g), torch.randn(6, 3, generator=g)
        opt.zero_grad()
        F.mse_loss(net(x), y).backward()
        opt.step()
    return dict((n, p.detach().clone()) for n, p in net.named_parameters())


for kw in [dict(), dict(momentum=0.9), dict(weight_decay=0.1),
           dict(momentum=0.9, weight_decay=0.1), dict(momentum=0.9, nesterov=True),
           dict(momentum=0.9, weight_decay=0.1, nesterov=True)]:
    mine, ref = sgd_trajectory(SGD, **kw), sgd_trajectory(torch.optim.SGD, **kw)
    for n in ref:
        assert torch.allclose(mine[n], ref[n], atol=1e-5), \
            f"SGD({kw}) diverges from torch.optim.SGD on {n} after 20 steps"
    assert torch.equal(mine["unused"], make_net().unused), "a parameter without .grad must not move"
print("SGD matches torch.optim.SGD for every option combination")

In [ ]:
def warmup_cosine_lr(step, total_steps, warmup_steps, base_lr, min_lr=0.0):
    """Linear warmup (step 0 -> base_lr / warmup_steps) then cosine decay to min_lr.

        step < warmup:          base_lr * (step + 1) / warmup
        warmup <= step < total: min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(pi * progress)),
                                progress = (step - warmup) / (total - warmup)  in [0, 1)
        step >= total:          min_lr
    """
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    if step >= total_steps:
        return min_lr                    # also avoids 0/0 when total_steps == warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
T, W, BASE, LO = 110, 10, 0.4, 0.01
lr = lambda s: float(warmup_cosine_lr(s, T, W, BASE, LO))
assert math.isclose(lr(0), BASE / W), f"step 0 should give base_lr/warmup = {BASE / W}, got {lr(0)}"
assert math.isclose(lr(W - 1), BASE), "the last warmup step reaches base_lr"
assert math.isclose(lr(W), BASE), "the cosine starts at base_lr at step == warmup_steps"
assert math.isclose(lr(W + (T - W) // 2), (BASE + LO) / 2), "cosine midpoint = (base + min) / 2"
assert math.isclose(lr(T), LO) and math.isclose(lr(T + 100), LO), "clamped to min_lr at/after the end"
sched = [lr(s) for s in range(T + 20)]
assert all(b > a for a, b in zip(sched[:W - 1], sched[1:W])), "strictly increasing warmup"
assert all(b <= a for a, b in zip(sched[W:], sched[W + 1:])), "non-increasing after warmup"
assert math.isclose(warmup_cosine_lr(0, 50, 0, 1.0), 1.0), "no warmup -> start at base_lr"
print("lr every 10 steps:", " ".join(f"{v:.3f}" for v in sched[::10]))

In [ ]:
@torch.no_grad()
def clip_grad_norm_(params, max_norm):
    """Clip gradients in place by their global L2 norm (torch.nn.utils.clip_grad_norm_ semantics).

    Returns the 0-dim total norm before clipping.
    """
    if isinstance(params, torch.Tensor):
        params = [params]
    grads = [p.grad for p in params if p.grad is not None]   # ONE pass: params may be a generator
    if not grads:
        return torch.tensor(0.0)
    # ||all grads||_2 = sqrt(sum_p ||g_p||^2) = || (||g_1||, ..., ||g_n||) ||_2
    total_norm = torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g) for g in grads]))
    clip_coef = max_norm / (total_norm + 1e-6)      # eps: zero grads give a huge coef, not 0/0
    if clip_coef < 1:
        for g in grads:
            g.mul_(clip_coef)                       # same factor for every tensor: direction kept
    return total_norm

In [ ]:
def net_with_grads(scale, seed=0):
    net = make_net(seed)
    for n, p in net.named_parameters():
        p.grad = None if n == "unused" else torch.randn_like(p) * scale
    return net

for scale, max_norm in [(3.0, 1.0), (0.01, 5.0)]:          # one clipped case, one untouched case
    mine, ref = net_with_grads(scale), net_with_grads(scale)
    got = clip_grad_norm_(mine.parameters(), max_norm)        # generator on purpose
    want = torch.nn.utils.clip_grad_norm_(ref.parameters(), max_norm)
    assert math.isclose(float(got), float(want), rel_tol=1e-5), f"norm {float(got)} != {float(want)}"
    for p, q in zip(mine.parameters(), ref.parameters()):
        assert (p.grad is None) == (q.grad is None)
        if q.grad is not None:
            assert torch.allclose(p.grad, q.grad, atol=1e-7), "clipped grads differ from PyTorch"

zero = make_net()
for p in zero.parameters():
    p.grad = torch.zeros_like(p)
assert float(clip_grad_norm_(zero.parameters(), 1.0)) == 0.0
assert all(torch.isfinite(p.grad).all() for p in zero.parameters()), "zero grads must not become NaN"
print("clip_grad_norm_ matches torch.nn.utils.clip_grad_norm_")

In [ ]:
class EMA:
    """Exponential moving average of a model's weights (buffers are copied, not averaged)."""

    def __init__(self, model, decay):
        self.decay = decay
        # detached clones: the shadow must neither share storage with nor track grads of the model
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters()}
        self.buffers = {n: b.detach().clone() for n, b in model.named_buffers()}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            # shadow = decay * shadow + (1 - decay) * p   (same as shadow.lerp_(p, 1 - decay))
            self.shadow[n].mul_(self.decay).add_(p, alpha=1 - self.decay)
        for n, b in model.named_buffers():
            self.buffers[n].copy_(b)          # running stats / num_batches_tracked: latest value

    @torch.no_grad()
    def copy_to(self, model):
        for n, p in model.named_parameters():
            p.copy_(self.shadow[n])           # in place: p stays the same trainable Parameter
        for n, b in model.named_buffers():
            b.copy_(self.buffers[n])

In [ ]:
torch.manual_seed(0)
bn_net = nn.Sequential(nn.Linear(4, 6), nn.BatchNorm1d(6), nn.ReLU(), nn.Linear(6, 2))
decay = 0.9
expected = {n: p.detach().clone() for n, p in bn_net.named_parameters()}   # theta_0
ema = EMA(bn_net, decay)
for k in range(5):
    with torch.no_grad():
        for p in bn_net.parameters():
            p.add_(torch.randn_like(p))       # pretend an optimizer step happened
    bn_net(torch.randn(8, 4) * 2 + 1)         # train-mode forward moves BN running stats
    ema.update(bn_net)
    for n, p in bn_net.named_parameters():
        expected[n] = decay * expected[n] + (1 - decay) * p.detach()

target = copy.deepcopy(bn_net)
ema.copy_to(target)
for n, p in target.named_parameters():
    assert torch.allclose(p, expected[n], atol=1e-6), f"EMA of {n} is not the closed form"
    assert isinstance(p, nn.Parameter) and p.requires_grad and p.grad_fn is None
for (n, b), b_live in zip(target.named_buffers(), bn_net.buffers()):
    assert torch.equal(b, b_live), f"buffer {n} must be copied from the model, not averaged"
print("EMA matches the closed form; buffers copied")

In [ ]:
# Putting it together: forward -> backward -> clip -> step -> schedule -> EMA, once per step.
def set_lr(optimizer, lr):
    for group in getattr(optimizer, "param_groups", []):   # torch.optim.Optimizer subclass
        group["lr"] = lr
    if hasattr(optimizer, "lr"):                             # plain class storing self.lr
        optimizer.lr = lr


def train_demo(min_lr, total=400, warmup=40, base_lr=0.1, verbose=False):
    torch.manual_seed(0)
    X = torch.randn(512, 5)
    Y = torch.sin(X @ torch.randn(5, 3)) + 0.05 * torch.randn(512, 3)
    model = nn.Sequential(nn.Linear(5, 64), nn.Tanh(), nn.Linear(64, 3))
    opt = SGD(model.parameters(), lr=0.0, momentum=0.9, weight_decay=1e-4, nesterov=True)
    ema = EMA(model, decay=0.99)                      # averages over ~1/(1-0.99) = 100 steps
    for step in range(total):
        lr = warmup_cosine_lr(step, total, warmup, base_lr, min_lr)
        set_lr(opt, lr)
        idx = torch.randint(0, 512, (32,))
        opt.zero_grad()
        loss = F.mse_loss(model(X[idx]), Y[idx])
        loss.backward()
        norm = clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        ema.update(model)
        if verbose and (step % 100 == 0 or step == total - 1):
            print(f"  step {step:3d}  lr {lr:.4f}  loss {loss.item():.4f}  grad-norm {float(norm):.3f}")
    ema_model = copy.deepcopy(model)
    ema.copy_to(ema_model)
    with torch.no_grad():
        return F.mse_loss(model(X), Y).item(), F.mse_loss(ema_model(X), Y).item()


for name, min_lr in [("cosine to 1e-3", 1e-3), ("warmup then constant", 0.1)]:
    online, averaged = train_demo(min_lr, verbose=(min_lr == 1e-3))
    print(f"{name:>22}:  full-data MSE online {online:.4f} | EMA {averaged:.4f}")
# Takeaway: when the LR is annealed to ~0 the iterates already settle and the EMA adds nothing
# (here it is even slightly worse: its window still contains weights from higher-LR steps).
# When the LR stays high (diffusion models often train at a constant LR), averaging the
# noisy iterates gives a clearly better model.

### Exercise (not graded): gradient accumulation

When a batch of 64 does not fit in memory, you run 4 micro-batches of 16, call `backward()` on each **without** zeroing the gradients in between, and step once. `.grad` *accumulates* (`+=`) across `backward()` calls. Implement `accumulated_grads` so the accumulated gradient equals the big-batch gradient exactly. How should each micro-batch loss be scaled, given that `nn.CrossEntropyLoss` averages over its batch?

Then run the BatchNorm cell below it. The equality breaks there. Why? And what else would break it (unequal micro-batch sizes, token-level losses with padding, dropout)?

In [ ]:
def big_batch_grads(model, X, y, loss_fn):
    model.zero_grad()
    loss_fn(model(X), y).backward()
    return [p.grad.clone() for p in model.parameters()]


def accumulated_grads(model, X, y, loss_fn, n_micro=4):
    """Same gradient as one big batch, computed from n_micro equal-size micro-batches."""
    model.zero_grad()
    for xb, yb in zip(X.chunk(n_micro), y.chunk(n_micro)):
        # mean over 64 = (1/4) * sum of the 4 means over 16 (equal sizes). For unequal sizes,
        # scale by len(xb) / len(X) instead.
        (loss_fn(model(xb), yb) / n_micro).backward()     # grads add up in .grad
    return [p.grad.clone() for p in model.parameters()]

In [ ]:
torch.manual_seed(0)
X_acc, y_acc = torch.randn(64, 10), torch.randint(0, 3, (64,))
ce = nn.CrossEntropyLoss()
mlp = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
for a, b in zip(big_batch_grads(mlp, X_acc, y_acc, ce), accumulated_grads(mlp, X_acc, y_acc, ce)):
    assert torch.allclose(a, b, atol=1e-6), "accumulated gradient != big-batch gradient"
print("MLP: 4 micro-batches with loss/4 == one batch of 64")

bn_mlp = nn.Sequential(nn.Linear(10, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 3))
diff = max((a - b).abs().max().item() for a, b in zip(big_batch_grads(bn_mlp, X_acc, y_acc, ce),
                                                      accumulated_grads(bn_mlp, X_acc, y_acc, ce)))
print(f"with BatchNorm: max |grad difference| = {diff:.3e}   <- why is this not ~0?")

**Answer.** For a mean-reduced loss over equal micro-batches,
$\frac{1}{64}\sum_{i=1}^{64}\ell_i = \frac14\sum_{m=1}^{4}\frac{1}{16}\sum_{i\in m}\ell_i$. Gradients are linear, so backpropagating `loss_m / 4` four times into the same `.grad` gives exactly the big-batch gradient (up to float rounding).

**BatchNorm breaks the equality.** Each sample's output depends on the mean and variance **of its own batch**. With micro-batches, BN normalises each group of 16 with that group's statistics. That is a different function from normalising all 64 together, so the losses differ and so do the gradients. The running statistics also get updated 4 times with noisier estimates instead of once. Fixes: use a normalisation that does not mix samples (LayerNorm/GroupNorm, as in most modern vision backbones and all transformers), freeze BN (eval mode, common when fine-tuning detectors with tiny per-GPU batches), or use SyncBatchNorm across devices. Accumulation does not help with this problem.

Other things that break "accumulation == big batch":
- **Unequal micro-batch sizes.** Scale by $n_m / N$, not $1/k$.
- **Token-level losses with padding** (LLM / captioning). Each micro-batch has a different number of real tokens, so "mean per micro-batch, then divide by $k$" ≠ the mean over all tokens. Normalise by the total token count across all micro-batches.
- **Dropout.** It draws different masks, so the gradients are equal in expectation but not bitwise.
- **Optimizer-step-based logic** (LR schedule, EMA, clipping) must run once per *optimizer* step, not once per micro-batch, and clipping must use the accumulated gradient.

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("training-utilities", SGD=SGD, warmup_cosine_lr=warmup_cosine_lr,
      clip_grad_norm_=clip_grad_norm_, EMA=EMA)

### Why it works

- **PyTorch's momentum convention.** The textbook form $v \leftarrow \mu v - \eta g,\ \theta \leftarrow \theta + v$ and PyTorch's $b \leftarrow \mu b + g,\ \theta \leftarrow \theta - \eta b$ are the same algorithm for a constant learning rate ($v = -\eta b$). They differ when $\eta$ changes, as it does under a schedule. PyTorch rescales the whole accumulated velocity by the *new* LR, while the textbook form keeps the old velocity at the old LR. Initialising $b_1 = g_1$ is identical to a zero buffer followed by $b = \mu\cdot 0 + g$. Adding a `(1 - μ)` dampening factor would shrink the steady-state step by a factor $1-\mu$ (10× smaller for μ = 0.9).
- **Nesterov.** The "look-ahead" gradient $\nabla L(\theta - \eta\mu b)$ is rewritten (Sutskever et al. 2013, Bengio et al. 2013) in terms of the current parameters, which gives $d = g + \mu b_t$: the current gradient plus a step further along the updated velocity. No second gradient evaluation is needed.
- **Weight decay inside the gradient** is exactly the gradient of $\frac{\lambda}{2}\|\theta\|^2$ (L2 regularisation), so it passes through momentum like any other gradient term. For plain SGD this equals *decoupled* decay $\theta \leftarrow \theta - \eta\lambda\theta$. With adaptive optimizers it does not (see the follow-ups).
- **Warmup + cosine.** Near initialisation, gradients are large and poorly conditioned, and Adam's second-moment estimates are still unreliable. A small LR for the first steps prevents early divergence, which matters most for transformers and for large-batch training (Goyal et al. 2017 pair the linear-scaling rule with warmup). Cosine annealing (SGDR, Loshchilov & Hutter 2017) lowers the LR smoothly so the final iterates settle into a minimum. Clamping at `min_lr` keeps the schedule well defined if training runs past `total_steps`.
- **Global-norm clipping** multiplies *all* gradients by one scalar ≤ 1. The update direction is preserved and only its length is capped. Per-element clipping (`clip_grad_value_`) and per-tensor clipping both change the direction. It protects against rare huge gradients: exploding RNN gradients (Pascanu et al. 2013) and loss spikes in LLM pretraining. The returned pre-clip norm is one of the most useful training-health metrics to log.
- **EMA** is a low-pass filter over the weight trajectory, with an effective window of about $1/(1-\text{decay})$ steps. Late in training, SGD iterates bounce around a minimum, and their average sits closer to its centre (Polyak–Ruppert averaging). That is why sampling from EMA weights gives visibly better diffusion samples. Buffers are copied rather than averaged because they are statistics of the *current* weights. Averaging BN running stats of different networks has no clear meaning, and `num_batches_tracked` is an integer. (timm's `ModelEmaV2` averages the whole `state_dict`, buffers included. Both conventions exist; this problem copies.)

### Complexity

All four utilities are $O(P)$ time per step for $P$ parameters. Momentum SGD keeps one extra buffer per parameter ($P$ floats), and so does the EMA (one more full copy of the model). Clipping makes two passes (a norm reduction, then a scale). On a GPU, the `if clip_coef < 1` branch above forces a host–device synchronisation. PyTorch avoids it by always multiplying by `clip_coef.clamp(max=1.0)`. PyTorch's `foreach` / `fused` implementations batch these per-tensor loops into a few multi-tensor kernels, which is a large win for models with thousands of small tensors.

### Common bugs

- Momentum with dampening `b = μb + (1−μ)g` (that is an EMA of gradients, not PyTorch SGD), or applying the LR inside the buffer.
- Weight decay applied *after* momentum, applied to parameters without gradients, or written in place into `p.grad`.
- A first momentum buffer that is `p.grad` itself instead of a clone, which `zero_grad(set_to_none=False)` silently zeroes.
- Storing `model.parameters()` without `list()`. The generator is exhausted after the first pass, so `step()` (or the scaling pass of clipping) silently does nothing.
- LR schedule: an off-by-one at step 0 (relative to the stated convention), running the cosine over `total_steps` instead of the post-warmup span, no clamp past the end (cosine climbs back up), or decaying to 0 instead of `min_lr`.
- Clipping per parameter, returning the *post*-clip norm, normalising small gradients up to `max_norm`, or $0/0$ = NaN for zero gradients. Under AMP, clipping before `scaler.unscale_(optimizer)` compares the *scaled* gradients against `max_norm`.
- EMA: storing references instead of clones (the "average" is then just the live weights), updating without `torch.no_grad()` (every update keeps the previous graph alive), averaging integer buffers, `copy_to` by `p.data = shadow` (shares storage, so training the model silently edits the EMA), and evaluating the online weights instead of the EMA weights.

### Interview follow-ups

1. **L2 regularisation vs decoupled weight decay: when are they different?** For SGD without momentum they are identical. With Adam, the L2 term is divided by $\sqrt{\hat v}$, so weights with large gradient variance are barely regularised. AdamW (Loshchilov & Hutter 2019) applies $\theta \leftarrow \theta - \eta\lambda\theta$ outside the adaptive step. In practice, biases and normalisation weights are usually excluded from decay through a separate param group.
2. **How do you clip gradients under mixed precision or in FSDP?** With `GradScaler`, call `scaler.unscale_(optimizer)` before `clip_grad_norm_` so that `max_norm` is in true units. `scaler.step` then skips the step if the gradients contain inf/NaN. With sharded parameters (FSDP/ZeRO), each rank computes the squared norm of its shard and an all-reduce sums them, because clipping by the local norm would give every rank a different scale. That is why FSDP has its own `clip_grad_norm_`.
3. **Why does the EMA need its own warmup?** With decay 0.9999, the shadow is dominated by the random initial weights for roughly the first 10k steps. Common fixes: a ramped decay such as $\min(\beta, \frac{1+t}{10+t})$ (TensorFlow's `ExponentialMovingAverage` with `num_updates`), starting the EMA after some steps, or bias correction like Adam's. Bias correction only works if the shadow starts at zero.
4. **Why warm up at all, and how long?** Early updates can be huge relative to the weights. With Adam, the second-moment estimate is computed from very few samples, so the effective step size has a high variance (the motivation behind RAdam). Warmup is typically ~1–5% of training, or a few thousand steps for LLMs. It also lets you use the linear-scaling rule ($\eta \propto$ batch size) without diverging.
5. **Why cosine rather than step decay, and what is WSD?** Cosine has no step boundaries to tune and decays smoothly. Its drawback is that you must know `total_steps` in advance. Warmup–Stable–Decay (constant LR, then a short final decay) lets you branch a decayed checkpoint off at any point, which is useful for continual pretraining and scaling-law sweeps.
6. **How is EMA related to MoCo/BYOL/DINO teachers?** Those "momentum encoders" are an EMA of the student's weights with decay ≈ 0.99–0.999. The slowly moving target network gives consistent targets, and without it the self-distillation objective collapses or oscillates.